In [1]:
import nltk
nltk.download("book")
from nltk.book import *

[nltk_data] Downloading collection 'book'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/abc.zip.
[nltk_data]    | Downloading package brown to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/brown.zip.
[nltk_data]    | Downloading package chat80 to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/chat80.zip.
[nltk_data]    | Downloading package cmudict to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/cmudict.zip.
[nltk_data]    | Downloading package conll2000 to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/conll2000.zip.
[nltk_data]    | Downloading package conll2002 to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/conll2002.zip.
[nltk_data]    | Downloading package dependency_treebank to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping corpora/dependency_treebank.zip.
[nltk_data]    | Downloading package genesis to /root/nltk_data...
[nltk_data]    

*** Introductory Examples for the NLTK Book ***
Loading text1, ..., text9 and sent1, ..., sent9
Type the name of the text or sentence to view it.
Type: 'texts()' or 'sents()' to list the materials.
text1: Moby Dick by Herman Melville 1851
text2: Sense and Sensibility by Jane Austen 1811
text3: The Book of Genesis
text4: Inaugural Address Corpus
text5: Chat Corpus
text6: Monty Python and the Holy Grail
text7: Wall Street Journal
text8: Personals Corpus
text9: The Man Who Was Thursday by G . K . Chesterton 1908


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

In [3]:
text1

<Text: Moby Dick by Herman Melville 1851>

In [4]:
bigramas = list(bigrams(text1))
bigramas[:10]

[('[', 'Moby'),
 ('Moby', 'Dick'),
 ('Dick', 'by'),
 ('by', 'Herman'),
 ('Herman', 'Melville'),
 ('Melville', '1851'),
 ('1851', ']'),
 (']', 'ETYMOLOGY'),
 ('ETYMOLOGY', '.'),
 ('.', '(')]

In [5]:
min_letras = 2
bigramas_filtrado = [b for b in bigramas if len(b[0]) > min_letras and len(b[1]) > min_letras ]
bigramas_filtrado_dist = FreqDist(bigramas_filtrado)

In [6]:
palabras_filtradas = [palabra for palabra in text1 if len(palabra) > min_letras]
palabras_filtradas_dist = FreqDist(palabras_filtradas)

# Dataframe para facilitar el análisis

In [7]:
df = pd.DataFrame()

df["bi_gram"] = list(set(bigramas_filtrado_dist))
df["word_0"] = df["bi_gram"].apply(lambda x: x[0])
df["word_1"] = df["bi_gram"].apply(lambda x: x[1])

df["bi_gram_freq"] = df["bi_gram"].apply(lambda x: bigramas_filtrado_dist[x])
df["word_0_freq"] = df["word_0"].apply(lambda x: palabras_filtradas_dist[x])
df["word_1_freq"] = df["word_1"].apply(lambda x: palabras_filtradas_dist[x])

df

,bi_gram,word_0,word_1,bi_gram_freq,word_0_freq,word_1_freq
0,"(preserve, much)",preserve,much,1,5,218
1,"(disastrous, encounter)",disastrous,encounter,1,3,16
2,"(clouds, whence)",clouds,whence,1,12,14
3,"(drama, was)",drama,was,1,3,1632
4,"(and, threes)",and,threes,3,6024,3
...,...,...,...,...,...,...
67937,"(heads, !""--)",heads,"!""--",1,85,13
67938,"(the, blunted)",the,blunted,1,13721,1
67939,"(the, wrought)",the,wrought,1,13721,6
67940,"(some, extraordinary)",some,extraordinary,1,578,7


# Pointwise Mutual Information (PMI)

Metrica que indica que tan probable es que dos palabras aparezcan juntas en comparación con lo que esperaríamos si fueran independientes.
**Detectar colocaciones**

In [8]:
df["PMI"] = (df[["bi_gram_freq", "word_0_freq", "word_1_freq"]]
             .apply(
                 lambda x: np.log2(x.values[0]/(x.values[1] * x.values[2])),
                 axis=1
              )
             )
df

,bi_gram,word_0,word_1,bi_gram_freq,word_0_freq,word_1_freq,PMI
0,"(preserve, much)",preserve,much,1,5,218,-10.090112
1,"(disastrous, encounter)",disastrous,encounter,1,3,16,-5.584963
2,"(clouds, whence)",clouds,whence,1,12,14,-7.392317
3,"(drama, was)",drama,was,1,3,1632,-12.257388
4,"(and, threes)",and,threes,3,6024,3,-12.556506
...,...,...,...,...,...,...,...
67937,"(heads, !""--)",heads,"!""--",1,85,13,-10.109831
67938,"(the, blunted)",the,blunted,1,13721,1,-13.744098
67939,"(the, wrought)",the,wrought,1,13721,6,-16.329061
67940,"(some, extraordinary)",some,extraordinary,1,578,7,-11.982281


In [9]:
df.sort_values(by="PMI", ascending=False)

,bi_gram,word_0,word_1,bi_gram_freq,word_0_freq,word_1_freq,PMI
49902,"(gudgeon, retires)",gudgeon,retires,1,1,1,0.000000
59896,"(NATHAN, COLEMAN)",NATHAN,COLEMAN,1,1,1,0.000000
56665,"(Fata, Morgana)",Fata,Morgana,1,1,1,0.000000
15361,"(Fitz, Swackhammer)",Fitz,Swackhammer,1,1,1,0.000000
53840,"(Fogo, Von)",Fogo,Von,1,1,1,0.000000
...,...,...,...,...,...,...,...
42154,"(man, the)",man,the,1,508,13721,-22.732783
55067,"(some, the)",some,the,1,578,13721,-22.919024
14670,"(one, the)",one,the,1,889,13721,-23.540138
15672,"(the, not)",the,not,1,13721,1103,-23.851315


In [10]:

df["log(bi_gram_freq)"] = df["bi_gram_freq"].apply(lambda x: np.log2(x))
df.sort_values(by="PMI", ascending=False)

,bi_gram,word_0,word_1,bi_gram_freq,word_0_freq,word_1_freq,PMI,log(bi_gram_freq)
49902,"(gudgeon, retires)",gudgeon,retires,1,1,1,0.000000,0.0
59896,"(NATHAN, COLEMAN)",NATHAN,COLEMAN,1,1,1,0.000000,0.0
56665,"(Fata, Morgana)",Fata,Morgana,1,1,1,0.000000,0.0
15361,"(Fitz, Swackhammer)",Fitz,Swackhammer,1,1,1,0.000000,0.0
53840,"(Fogo, Von)",Fogo,Von,1,1,1,0.000000,0.0
...,...,...,...,...,...,...,...,...
42154,"(man, the)",man,the,1,508,13721,-22.732783,0.0
55067,"(some, the)",some,the,1,578,13721,-22.919024,0.0
14670,"(one, the)",one,the,1,889,13721,-23.540138,0.0
15672,"(the, not)",the,not,1,13721,1103,-23.851315,0.0


In [19]:
# x = PMI + log(bi_gram_freq)
# x es grande -> 0
# x es pequeño -> 1
(df["PMI"]+df["log(bi_gram_freq)"]).apply(lambda x: 1/(1+abs(x)))

,0
0,0.090170
1,0.151861
2,0.119157
3,0.075430
4,0.083531
...,...
67937,0.090010
67938,0.067824
67939,0.057707
67940,0.077028


**Gráfico de PMI vs Log(bi_gram_freq)**

In [23]:
fig = px.scatter(
      x=df["PMI"].values, y=df["log(bi_gram_freq)"].values,
      color=df["PMI"]+df["log(bi_gram_freq)"],
      size=(df["PMI"]+df["log(bi_gram_freq)"]).apply(lambda x: 1/(1+abs(x))).values,
      hover_name=df["bi_gram"].values,
      labels={"x":"PMI", "y": "Freq"},
      width=800, height=600
    )
fig.show()

Output hidden; open in https://colab.research.google.com to view.

# PMI con NLTK

In [25]:
from nltk.collocations import *

In [32]:
# https://tedboy.github.io/nlps/generated/generated/nltk.BigramAssocMeasures.html
bigram_measures = nltk.collocations.BigramAssocMeasures()
# https://www.nltk.org/api/nltk.collocations.BigramCollocationFinder.html
finder = nltk.collocations.BigramCollocationFinder.from_words(text1)

In [ ]:
finder.apply_freq_filter(20)
finder.nbest(bigram_measures.pmi, 10)

### Ejercicio: Encontrar 10 colocaciones en cees_esp

In [ ]:
nltk.download("cess_esp")
corpus = nltk.corpus.cess_esp.sents()
corpus

In [ ]:
words = [w for l in corpus for w in l]
words

In [ ]:
finder = nltk.collocations.BigramCollocationFinder.from_words(words)
finder.apply_freq_filter(20)
finder.nbest(bigram_measures.pmi, 10)

In [45]:
finder = nltk.collocations.BigramCollocationFinder.from_documents(corpus)
finder.apply_freq_filter(20)
finder.nbest(bigram_measures.pmi, 10)

[('secretario', 'general'),
 ('primer', 'ministro'),
 ('informó', 'hoy'),
 ('año', 'pasado'),
 ('este', 'año'),
 ('cinco', 'años'),
 ('ha', 'sido'),
 ('desde', 'hace'),
 ('han', 'sido'),
 ('puede', 'ser')]